# HandsOn HO-3 RDD Spark
Selamat datang di HandsOn HO-3, yaitu tentang pemrosesan data terdistribusi menggunakan Spark. Untuk tujuan pembelajaran, seperti biasa, kita akan menggunakan *pseudo-distributed mode* (single node cluster) di VM yang telah disediakan. Dengan kode yang *similar* di cluster komputer dengan *n* workers, maka komputasi akan tersebar ke *n* workers tersebut. Adapun yang akan kita coba kali ini adalah melakukan komputasi menggunakan RDD dan DataFrame. Berikut catatan-catatan yang perlu kamu perhatikan dalam hands-on ini:
1. Untuk menjalankan Apache Spark dalam bahasa python di VM, ketikkan perintah ```pyspark``` di terminal.
2. Dari semua Milestone, data input yang digunakan adalah data "purchases.txt" yang telah diletakkan di HDFS. Oleh karena itu, pastikan hadoop service kamu berjalan (```start-dfs.sh```, ```start-yarn.sh```, ```jps```). Untuk membaca data dari HDFS, lihat kembali di slide perkuliahan.
3. Untuk Milestone 1, 2 dan 3, kalian perlu untuk mencatat waktu yang diperlukan saat melakukan MapReduce menggunakan hadoop streaming jar di hands-on sebelumnya. Waktu bisa dihitung dari selisih "waktu awal" dan "waktu akhir" yang tampak di terminal saat kalian selesain melakukan MapReduce -atau menggunakan cara lain yang masih *acceptable*-. (lihat ilustrasi di bawah).
4. Lakukan zip file jupyter notebook ini beserta gambar-gambar yang diperlukan -screenshot waktu proses MapReduce Hadoop jar-, dan submit ke portal kuliah EDUNEX dengan format nama "**HandsOn_HO3_NIM_NamaLengkap.zip**". Pastikan file jupyter notebook yang kamu zip dalam kondisi memiliki output per cellnya (tidak kosong karena belum dijalankan). <br>

<img title="Waktu Awal" align="left" src="waktu_awal.JPG" alt="Drawing" style="width: 600px;"/>
<img title="Waktu Akhir" align="left" src="waktu_akhir.JPG" alt="Drawing" style="width: 600px;"/>

## Milestone 1
Kerjakan Milestone 1 pada HandsOn HO2 (sebelumnya), akan tetapi menggunakan RDD Spark. Catat waktu (bandingkan) yang dibutuhkan (dalam detik) antara: "MapReduce menggunakan hadoop streaming jar" dengan yang akan kamu proses menggunakan RDD Spark ini.

In [1]:
from time import time
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('HO3-RDD').getOrCreate()
sc = spark.sparkContext

# Lokasi file di HDFS
DATA_PATH = 'hdfs://namenode:9000/data/purchases/purchases.txt'

# RDD dasar
lines = sc.textFile(DATA_PATH)
fields_rdd = lines.map(lambda l: l.split('	'))


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/24 07:18:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


#### {tempatkan gambar screenshot yang menunjukkan waktu proses MapReduce Hadoop jar di sini}

<img title="Waktu Awal" align="left" src="hadoop1.png" alt="Drawing" style="width: 600px;"/>

## Milestone 2
Kerjakan Milestone 2 pada HandsOn HO2 (sebelumnya), akan tetapi menggunakan RDD Spark. Catat waktu (bandingkan) yang dibutuhkan (dalam detik) antara: "MapReduce menggunakan hadoop streaming jar" dengan yang akan kamu proses menggunakan RDD Spark ini.

In [2]:
# Milestone 1 — total transaksi untuk kategori Toys dan Consumer Electronics (RDD)
time_m1_hadoop = 3.177  # isi manual hasil Hadoop streaming sebelumnya

start = time()
totals_m1 = (
    fields_rdd
    .filter(lambda f: f[3] in ('Toys', 'Consumer Electronics'))
    .map(lambda f: (f[3], float(f[4])))
    .reduceByKey(lambda a, b: a + b)
    .collectAsMap()
)
time_m1_rdd = time() - start

print('Total per kategori (RDD):', totals_m1)
print('Waktu Hadoop:', time_m1_hadoop)
print('Waktu RDD:', time_m1_rdd)


[Stage 0:>                                                          (0 + 2) / 2]

Total per kategori (RDD): {'Consumer Electronics': 57452374.130000055, 'Toys': 57463477.10999993}
Waktu Hadoop: 3.177
Waktu RDD: 2.0593814849853516


#### {tempatkan gambar screenshot yang menunjukkan waktu proses MapReduce Hadoop jar di sini}
<img title="Waktu Awal" align="left" src="hadoop2.png" alt="Drawing" style="width: 600px;"/>

## Milestone 3
Kerjakan Milestone 3 pada HandsOn HO2 (sebelumnya), akan tetapi menggunakan RDD Spark. Catat waktu (bandingkan) yang dibutuhkan (dalam detik) antara: "MapReduce menggunakan hadoop streaming jar" dengan yang akan kamu proses menggunakan RDD Spark ini.

In [3]:
# Milestone 2 — harga maksimum untuk Miami, San Francisco, Atlanta (RDD)
time_m2_hadoop = 7.265  # isi manual hasil Hadoop streaming sebelumnya

cities = {'Miami', 'San Francisco', 'Atlanta'}

start = time()
max_by_city = (
    fields_rdd
    .filter(lambda f: f[2] in cities)
    .map(lambda f: (f[2], (float(f[4]), f[3])))
    .reduceByKey(lambda a, b: a if a[0] >= b[0] else b)
    .collect()
)
time_m2_rdd = time() - start

for city, (price, product) in sorted(max_by_city):
    print(f'{city}: {price} — {product}')
print('Waktu Hadoop:', time_m2_hadoop)
print('Waktu RDD:', time_m2_rdd)


[Stage 2:>                                                          (0 + 2) / 2]

Atlanta: 499.96 — Pet Supplies
Miami: 499.98 — Video Games
San Francisco: 499.97 — Men's Clothing
Waktu Hadoop: 7.265
Waktu RDD: 1.3148109912872314


#### {tempatkan gambar screenshot yang menunjukkan waktu proses MapReduce Hadoop jar di sini}
<img title="Waktu Awal" align="left" src="hadoop3.png" alt="Drawing" style="width: 600px;"/>

## Milestone 4
Milestone ini dibagi menjadi 4.1, 4.2 dan 4.3 yang masing-masing secara berturut-turut adalah mengerjakan ulang Milestone 1, 2 dan 3 di atas (menggunakan RDD Spark), akan tetapi menggunakan trik "**persist() RDD**" untuk mempercepat prosesnya. Kamu bisa melakukan "**persist**" untuk RDD mana saja yang kamu anggap dapat memberikan waktu proses tercepat.

In [4]:
# Milestone 3 — menghitung jumlah transaksi pada rentang waktu tertentu (RDD)
time_m3_hadoop = 145.146  # isi manual hasil Hadoop streaming sebelumnya


def bucket_time(f):
    hour, minute = map(int, f[1].split(':'))
    if (hour == 9 and minute >= 1) or (hour == 10 and minute == 0):
        return '09:01-10:00'
    if (hour == 10 and minute >= 1) or (hour == 11 and minute == 0):
        return '10:01-11:00'
    return None

start = time()
time_ranges = (
    fields_rdd
    .map(bucket_time)
    .filter(lambda r: r is not None)
    .map(lambda r: (r, 1))
    .reduceByKey(lambda a, b: a + b)
    .collectAsMap()
)
time_m3_rdd = time() - start

print('Jumlah per rentang:', dict(sorted(time_ranges.items())))
print('Waktu Hadoop:', time_m3_hadoop)
print('Waktu RDD:', time_m3_rdd)


[Stage 4:>                                                          (0 + 2) / 2]

Jumlah per rentang: {'09:01-10:00': 459775, '10:01-11:00': 459825}
Waktu Hadoop: 145.146
Waktu RDD: 1.8286426067352295


## Milestone 5
Milestone ini dibagi menjadi 5.1, 5.2 dan 5.3 yang masing-masing secara berturut-turut adalah mengerjakan ulang Milestone 1, 2 dan 3 di atas, akan tetapi menggunakan DataFrame dari Apache Spark. Catat waktu yang diperlukan untuk masing-masing proses (5.1, 5.2 dan 5.3).

In [5]:
# Milestone 4 — ulangi dengan persist() pada RDD untuk mempercepat
cached = fields_rdd.persist()

start = time()
totals_m1_cached = (
    cached
    .filter(lambda f: f[3] in ('Toys', 'Consumer Electronics'))
    .map(lambda f: (f[3], float(f[4])))
    .reduceByKey(lambda a, b: a + b)
    .collectAsMap()
)
time_m4_1 = time() - start

start = time()
max_by_city_cached = (
    cached
    .filter(lambda f: f[2] in cities)
    .map(lambda f: (f[2], (float(f[4]), f[3])))
    .reduceByKey(lambda a, b: a if a[0] >= b[0] else b)
    .collect()
)
time_m4_2 = time() - start

start = time()
time_ranges_cached = (
    cached
    .map(bucket_time)
    .filter(lambda r: r is not None)
    .map(lambda r: (r, 1))
    .reduceByKey(lambda a, b: a + b)
    .collectAsMap()
)
time_m4_3 = time() - start

print('M1 cached:', totals_m1_cached)
print('M2 cached:', sorted(max_by_city_cached))
print('M3 cached:', dict(sorted(time_ranges_cached.items())))
print('Waktu M1 (Hadoop vs RDD vs cached):', time_m1_hadoop, time_m1_rdd, time_m4_1)
print('Waktu M2 (Hadoop vs RDD vs cached):', time_m2_hadoop, time_m2_rdd, time_m4_2)
print('Waktu M3 (Hadoop vs RDD vs cached):', time_m3_hadoop, time_m3_rdd, time_m4_3)


[Stage 10:>                                                         (0 + 2) / 2]

M1 cached: {'Consumer Electronics': 57452374.130000055, 'Toys': 57463477.10999993}
M2 cached: [('Atlanta', (499.96, 'Pet Supplies')), ('Miami', (499.98, 'Video Games')), ('San Francisco', (499.97, "Men's Clothing"))]
M3 cached: {'09:01-10:00': 459775, '10:01-11:00': 459825}
Waktu M1 (Hadoop vs RDD vs cached): 3.177 2.0593814849853516 2.778532028198242
Waktu M2 (Hadoop vs RDD vs cached): 7.265 1.3148109912872314 0.9421632289886475
Waktu M3 (Hadoop vs RDD vs cached): 145.146 1.8286426067352295 1.521655559539795
